In [1]:
import pandas as pd
import requests
import datetime as dt

In [2]:
def int_from_str(t):
    return int(t.replace(":",""))

In [3]:
def get_players_stats_per_game(base_url: str, player: str, seasons: list, game_type: str) -> pd.DataFrame:
    '''For given player, season and optional game_type parameters return stats for given player by game played'''
    master = []
    for season in seasons:
        response = requests.get(f'{base_url}/{player}/game-log/{season}/{game_type}', timeout=20)
        response.raise_for_status()

        game_log = response.json()
        master = master + game_log['gameLog']

    return pd.DataFrame(master)

In [4]:
def process_games_data(games: pd.DataFrame) -> pd.DataFrame:
    games['gameDate'] = pd.to_datetime(games['gameDate'])
    games = games.sort_values('gameDate')
    games['points_l5_games'] = games['points'].shift(1).rolling(5, min_periods=1).sum()
    games['goals_l5_games'] = games['goals'].shift(1).rolling(5, min_periods=1).sum()
    games['shots_l5_games'] = games['shots'].shift(1).rolling(5, min_periods=1).sum()
    games['home_game'] = [1 if x=='H' else 0 for x in games['homeRoadFlag']]
    return games

In [5]:
def process_standings_table(games: pd.DataFrame) -> pd.DataFrame:
    for index, row in games.iterrows():
        date = (pd.to_datetime(row['gameDate']) - dt.timedelta(days=1)).strftime('%Y-%m-%d')

        response = requests.get(url = f"https://api-web.nhle.com/v1/standings/{date}", timeout=30)
        response.raise_for_status()

        data = response.json()

        standings_df = pd.json_normalize(data["standings"])

        games.loc[index, 'oponent_goalAgainst'] = round(standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['goalAgainst'].iloc[0] / standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['gamesPlayed'].iloc[0], 5) 
        games.loc[index, 'oponent_goalFor'] = round(standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['goalFor'].iloc[0] /  standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['gamesPlayed'].iloc[0] , 5)
        games.loc[index, 'oponent_pointPctg'] = standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['pointPctg'].iloc[0]
    return games


In [6]:

def get_players_shifts_per_game(game_id: str) -> pd.DataFrame:

    '''
    Get shifts information for given game_id. 
    '''
    url = (
        "https://api.nhle.com/stats/rest/en/shiftcharts"
        f"?cayenneExp=gameId={game_id}"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    data = response.json()
    shifts = data.get("data", [])

    if shifts:
        shifts_df = pd.DataFrame(shifts)
        return shifts_df
    else:
        print(f"No shift-chart records returned for game {game_id}")
        return pd.DataFrame()
    


In [7]:
def get_processed_players_shifts_data(shifts_df : pd.DataFrame, player: str) -> pd.DataFrame:
    '''
    Filter data by target player presence. Add integer equivalent of the start and end time of the shift, to reject event-based data record whenever target player was not on ice  
    '''
    
    shifts_df = shifts_df[(shifts_df['playerId']== int(player))]

    shifts_df['overallstartTime'] = shifts_df.apply(lambda x: f"{int(x['startTime'][0]) + ((int(x['period']) - 1)*2)}{x['startTime'][1]}:{x['startTime'][3]}{x['startTime'][4]}", axis=1)
    shifts_df['overallendTime'] = shifts_df.apply(lambda x: f"{int(x['endTime'][0]) + ((int(x['period']) - 1)*2)}{x['endTime'][1]}:{x['endTime'][3]}{x['endTime'][4]}", axis=1)

    shifts_df['overallstartTime_int'] = shifts_df.overallstartTime.apply(lambda x: int_from_str(x))
    shifts_df['overallendTime_int'] = shifts_df.overallendTime.apply(lambda x: int_from_str(x))
    return shifts_df


In [8]:
def get_play_by_play_data(game_id: str) -> pd.DataFrame:
    '''
    Get game events data given game id 
    '''
    url = (
        f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    play_by_play = response.json()
    return play_by_play, pd.DataFrame(play_by_play['plays'])


In [9]:
def get_period_info(play_by_play_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Get information about period number and type
    '''
    play_by_play_df['periodNumber'] = play_by_play_df['periodDescriptor'].apply(lambda x: x['number'])
    play_by_play_df['periodType'] = play_by_play_df['periodDescriptor'].apply(lambda x: x['periodType'])
    return play_by_play_df

In [10]:
def get_events_info(play_by_play_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Add information about events
    '''
    unwanted_events = ['period-start', 'stoppage', 'period-end', 'game-end']

    play_by_play_df = play_by_play_df[~play_by_play_df['typeDescKey'].isin(unwanted_events)]

    keys = []
    for d in  play_by_play_df['details']:
        if type(d) == dict:
            keys = keys + list(d.keys())


    irrelevant_keys = [ 'discreteClip', 'discreteClipFr',  'highlightClip', 'highlightClipFr', 
                    'highlightClipSharingUrl', 'highlightClipSharingUrlFr',  'xCoord', 'yCoord']

    keys = set(keys).difference(irrelevant_keys)

    for k in keys:
        play_by_play_df[k] = play_by_play_df['details'].apply(lambda x: x[k] if type(x)==dict and k in x.keys() else None)
    return play_by_play_df

In [11]:
def get_team_specific_info(play_by_play_df: pd.DataFrame, play_by_play: dict, team_id: str) -> pd.DataFrame:
    '''
    add team-specific info (scored or allowed, PP or PK)
    '''
    for k in ['faceoff','takeaway','giveaway','shot-on-goal','blocked-shot','hit','missed-shot','goal','delayed-penalty','penalty']:
        play_by_play_df[k] = play_by_play_df.apply(lambda x: 1 if x['typeDescKey']==k and x['details']['eventOwnerTeamId']==int(team_id) else -1 if x['typeDescKey']==k and x['details']['eventOwnerTeamId']!=int(team_id) else 0, axis=1)

    play_by_play_df['oz_faceoff'] = play_by_play_df.apply(lambda x: 1 if x['typeDescKey']=='faceoff' and ((x['details']['eventOwnerTeamId']==int(team_id) and x['details']['zoneCode']=='O') or (x['details']['eventOwnerTeamId']!=int(team_id) and x['details']['zoneCode']=='D')) else \
                                                          -1 if x['typeDescKey']=='faceoff' and ((x['details']['eventOwnerTeamId']!=int(team_id) and x['details']['zoneCode']=='O') or (x['details']['eventOwnerTeamId']==int(team_id) and x['details']['zoneCode']=='D')) else 0, axis=1)


    # situationCode column tells us formations quantities, for example 0641 means that visiting team is in PP and is playing without a goalie 
    home_team = play_by_play['homeTeam']['id'] == int(team_id)

    play_by_play_df['PP'] = play_by_play_df['situationCode'].apply(lambda x: (int(x[0]) + int(x[1]) - int(x[2]) - int(x[3]))* -1 if home_team else  (int(x[0]) + int(x[1]) - int(x[2]) - int(x[3])))
    play_by_play_df['Goalie_Pulled'] = play_by_play_df['situationCode'].apply(lambda x: (int(x[0]) - int(x[3])) if home_team else  (int(x[0]) - int(x[3]))* -1)

    play_by_play_df = play_by_play_df.drop(columns = ['periodDescriptor','details', 'typeCode', 'timeRemaining','homeTeamDefendingSide', 'sortOrder', 'eventId', 
                                                    'situationCode', 'pptReplayUrl', 'shotType', 'descKey', 'scoringPlayerTotal', 'homeSOG', 'duration', 'awayScore',
                                                    'eventOwnerTeamId', 'homeScore', 'assist1PlayerTotal', 'awaySOG', 'goalieInNetId','assist2PlayerTotal', 'timeRemaining', 
                                                    'reason', 'goalInGame', 'servedByPlayerId'], errors="ignore")
    return play_by_play_df

In [12]:
def get_player_specific_info(play_by_play_df: pd.DataFrame, player: str) -> pd.DataFrame:
    '''
    add player-specific info (scored or allowed, PP or PK). player IDs for each event are not relevant unless it was our target player
    '''
    play_by_play_df['player_hit'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'hittingPlayerId' in x and int(x['hittingPlayerId']) == int(player) else -1 if (x['hitteePlayerId']) == int(player) else 0, axis=1)
    play_by_play_df['player_assist1'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'assist1PlayerId' in x and int(x['assist1PlayerId']) == int(player) else 0, axis=1)
    play_by_play_df['player_assist2'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'assist2PlayerId' in x and int(x['assist2PlayerId']) == int(player) else 0, axis=1)
    play_by_play_df['player_goal'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'scoringPlayerId' in x and int(x['scoringPlayerId']) == int(player) else 0, axis=1)
    play_by_play_df['player_drawn_penalty'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'drawnByPlayerId' in x  and int(x['drawnByPlayerId']) == int(player) else -1 if 'committedByPlayerId' in x and int(x['committedByPlayerId']) == int(player) else 0, axis=1)
    play_by_play_df['player_faceoff'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'winningPlayerId' in x and int(x['winningPlayerId']) == int(player) else -1 if int(x['losingPlayerId']) == int(player) else 0, axis=1)
    play_by_play_df['player_takeaway'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'playerId' in x and  (int(x['playerId']) == int(player) and x['typeDescKey']=='takeaway')  else -1 if (int(x['playerId']) == int(player) and x['typeDescKey']=='giveaway') else 0, axis=1)

    play_by_play_df['player_shot_on_goal'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'shootingPlayerId' in x and  (int(x['shootingPlayerId']) == int(player) and x['typeDescKey']=='shot-on-goal')  else 0, axis=1)
    play_by_play_df['player_blocked_shot'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'blockingPlayerId' in x and  (int(x['blockingPlayerId']) == int(player) and x['typeDescKey']=='blocked-shot') else 0, axis=1)
    play_by_play_df['player_missed_shot'] = play_by_play_df.fillna(0).apply(lambda x: 1 if 'shootingPlayerId' in x and  (int(x['shootingPlayerId']) == int(player) and x['typeDescKey']=='missed-shot') else 0, axis=1)

    play_by_play_df = play_by_play_df.drop(columns = ['assist2PlayerId', 'playerId', 'winningPlayerId', 'shootingPlayerId', 
                                                    'assist1PlayerId', 'drawnByPlayerId', 'blockingPlayerId', 'hittingPlayerId', 
                                                    'scoringPlayerId', 'hitteePlayerId', 'committedByPlayerId', 'losingPlayerId', 'shootout-complete', 'failed-shot-attempt'], errors="ignore")
    return play_by_play_df

In [13]:
def add_identifier_timestamps(play_by_play_df: pd.DataFrame, game_id: int) -> pd.DataFrame:
    play_by_play_df['game_id'] = game_id
    play_by_play_df['overalltimeInPeriod'] = play_by_play_df.apply(lambda x: f"{int(x['timeInPeriod'][0]) + ((int(x['periodNumber']) - 1)*2)}{x['timeInPeriod'][1]}:{x['timeInPeriod'][3]}{x['timeInPeriod'][4]}", axis=1)
    play_by_play_df['overalltimeInPeriod_int'] = play_by_play_df.overalltimeInPeriod.apply(lambda x: int_from_str(x))
    return play_by_play_df

In [14]:
def add_aggregated_info(play_by_play_df: pd.DataFrame):
    play_by_play_df["shot-on-goal_l5"] =  play_by_play_df["overalltimeInPeriod_int"].apply( lambda current: play_by_play_df.loc[(play_by_play_df["overalltimeInPeriod_int"] > current - 500) &(play_by_play_df["overalltimeInPeriod_int"] < current),"shot-on-goal"].sum())
    play_by_play_df["takeaways_l5"] =  play_by_play_df["overalltimeInPeriod_int"].apply( lambda current: play_by_play_df.loc[(play_by_play_df["overalltimeInPeriod_int"] > current - 500) &(play_by_play_df["overalltimeInPeriod_int"] < current),"takeaway"].sum())
    play_by_play_df["giveaways_l5"] =  play_by_play_df["overalltimeInPeriod_int"].apply( lambda current: play_by_play_df.loc[(play_by_play_df["overalltimeInPeriod_int"] > current - 500) &(play_by_play_df["overalltimeInPeriod_int"] < current),"giveaway"].sum())
    play_by_play_df['score_diff'] = play_by_play_df["goal"].cumsum() 
    return play_by_play_df

In [15]:
def filter_events_by_players_shifts(shifts_df: pd.DataFrame, play_by_play_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Cross-reference games events with shifts of the target player  
    '''


    for index, row in shifts_df.iterrows():
        df = play_by_play_df[(play_by_play_df['overalltimeInPeriod_int']>=row['overallstartTime_int']) & (play_by_play_df['overalltimeInPeriod_int']<=row['overallendTime_int'])]
        if len(df)>0:
            for c in set(['oz_faceoff','player_faceoff','takeaway','giveaway','player_takeaway','shot-on-goal','player_shot_on_goal','blocked-shot','player_blocked_shot','hit','player_hit','missed-shot','player_missed_shot','goal','player_goal','player_assist1','player_assist2']).intersection(list(df.columns)):
                shifts_df.loc[index, c] = df[c].sum()
            for c in set(['delayed-penalty','player_drawn_penalty','penalty','PP','Goalie_Pulled', 'shot-on-goal_l5', 'takeaways_l5', 'giveaways_l5', 'score_diff']).intersection(list(df.columns)):
                shifts_df.loc[index, c] = df[c].max()

    shifts_df = shifts_df.drop(columns = ['detailCode', 'eventDescription','eventDetails','eventNumber','hexValue','playerId','lastName','firstName','teamAbbrev','teamName','typeCode','overallstartTime_int', 'overallendTime_int'], errors="ignore")


    shifts_df['duration_m'] = shifts_df.apply(lambda x: round((int(x['endTime'][:2])*60 + int((x['endTime'][3:5])) - int(x['startTime'][:2])*60 - int((x['startTime'][3:5])))/60,5), axis=1)
    shifts_df['cumsum_icetime'] = shifts_df["duration_m"].cumsum() 
    
    return shifts_df

In [25]:
def merge_games_with_shifts(games, players_master_data):
    return pd.merge(games[['gameId','points_l5_games','goals_l5_games','shots_l5_games', 'oponent_goalAgainst', 'oponent_goalFor','oponent_pointPctg', 'home_game', 'gameDate']], players_master_data, how='right', on='gameId')

In [17]:
def process_players_data(base_url: str, player: str, seasons: list, game_type: str, team_id: str):
    games = get_players_stats_per_game(base_url, player, seasons, game_type)
    games = process_games_data(games)
    games = process_standings_table(games)
    
    players_master_data = pd.DataFrame()

    for game_id in games['gameId']:
        print(f'PROCESSING GAME ID {game_id}')
        players_master_data = pd.concat([players_master_data,process_single_game(game_id, player, team_id)])


    return games, players_master_data
        


In [18]:
def process_single_game(game_id, player, team_id):
    shifts_df = get_players_shifts_per_game(game_id)

    if shifts_df.empty:
        return pd.DataFrame()

    shifts_df = get_processed_players_shifts_data(shifts_df, player)

    play_by_play, play_by_play_df = get_play_by_play_data(game_id)
    play_by_play_df = get_period_info(play_by_play_df)
    play_by_play_df = get_events_info(play_by_play_df)
    play_by_play_df = get_team_specific_info(play_by_play_df, play_by_play, team_id)
    play_by_play_df = get_player_specific_info(play_by_play_df, player)
    play_by_play_df = add_identifier_timestamps(play_by_play_df, game_id)
    play_by_play_df = add_aggregated_info(play_by_play_df)
    shifts_df = filter_events_by_players_shifts(shifts_df, play_by_play_df)
      
    return shifts_df

In [19]:
games, shifts_data = process_players_data("https://api-web.nhle.com/v1/player", '8478402', ["20232024","20242025","20252026"], '2', '22')

/var/folders/0_/wk4545mj60g3lxt1n0vnfycc0000gn/T/ipykernel_10201/174838185.py:12: RuntimeWarning: invalid value encountered in long_scalars
  games.loc[index, 'oponent_goalAgainst'] = round(standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['goalAgainst'].iloc[0] / standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['gamesPlayed'].iloc[0], 5)
/var/folders/0_/wk4545mj60g3lxt1n0vnfycc0000gn/T/ipykernel_10201/174838185.py:13: RuntimeWarning: invalid value encountered in long_scalars
  games.loc[index, 'oponent_goalFor'] = round(standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['goalFor'].iloc[0] /  standings_df[standings_df['teamAbbrev.default']==row['opponentAbbrev']]['gamesPlayed'].iloc[0] , 5)


PROCESSING GAME ID 2023020009


/var/folders/0_/wk4545mj60g3lxt1n0vnfycc0000gn/T/ipykernel_10201/1057762501.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shifts_df['overallstartTime'] = shifts_df.apply(lambda x: f"{int(x['startTime'][0]) + ((int(x['period']) - 1)*2)}{x['startTime'][1]}:{x['startTime'][3]}{x['startTime'][4]}", axis=1)
/var/folders/0_/wk4545mj60g3lxt1n0vnfycc0000gn/T/ipykernel_10201/1057762501.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  shifts_df['overallendTime'] = shifts_df.apply(lambda x: f"{int(x['endTime'

PROCESSING GAME ID 2023020029
PROCESSING GAME ID 2023020044
PROCESSING GAME ID 2023020054
PROCESSING GAME ID 2023020077
PROCESSING GAME ID 2023020129
PROCESSING GAME ID 2023020152
PROCESSING GAME ID 2023020159
PROCESSING GAME ID 2023020179
PROCESSING GAME ID 2023020203
PROCESSING GAME ID 2023020220
PROCESSING GAME ID 2023020227
PROCESSING GAME ID 2023020239
PROCESSING GAME ID 2023020257
PROCESSING GAME ID 2023020272
PROCESSING GAME ID 2023020279
PROCESSING GAME ID 2023020298
PROCESSING GAME ID 2023020320
PROCESSING GAME ID 2023020335
PROCESSING GAME ID 2023020349
PROCESSING GAME ID 2023020391
PROCESSING GAME ID 2023020407
PROCESSING GAME ID 2023020421
PROCESSING GAME ID 2023020437
PROCESSING GAME ID 2023020452
PROCESSING GAME ID 2023020472
PROCESSING GAME ID 2023020490
PROCESSING GAME ID 2023020501
PROCESSING GAME ID 2023020511
PROCESSING GAME ID 2023020545
PROCESSING GAME ID 2023020563
PROCESSING GAME ID 2023020569
PROCESSING GAME ID 2023020583
PROCESSING GAME ID 2023020615
PROCESSING

In [20]:
games.to_csv('Data/players_games_data.csv', index=False)
shifts_data.to_csv('Data/shifts_data.csv', index=False)

In [26]:
shifts_data_merged = merge_games_with_shifts(games, shifts_data)
shifts_data_merged.to_csv('Data/model_input_data.csv', index=False)